### **we will programatically generate multiple of our own messy Data for testing (As Data Engineers Do).**

In [1]:
import pandas as pd
import numpy as np

# 1. Generate the core data (120 rows)
np.random.seed(42)
student_ids = [f"00{i}" for i in range(101, 221)]
names = [f"Student_{i}" for i in range(1, 121)]
courses = np.random.choice(['Data Science', 'DBMS', 'Web Scraping', 'Business Analytics'], 120)
statuses = np.random.choice(['Contacted', 'TBD', 'N/A', 'Enrolled'], 120)
scores = np.random.randint(40, 100, 120)

# Introduce some actual nulls and messy string values
statuses[10] = 'Missing'
statuses[45] = 'N/A'

# 2. Create the data grid
data_grid = {
    'Spacer': [''] * 120, # This simulates an empty Column A 
    'StudentID': student_ids,
    'FullName': names,
    'Course_Interest': courses,
    'Contact_Status': statuses,
    'Score': scores
}
df_data = pd.DataFrame(data_grid)

# 3. Create the messy human-readable header rows
header_rows = pd.DataFrame([
    ['', 'ACADEMY LEAD GENERATION REPORT', '', '', '', ''],
    ['', 'CONFIDENTIAL - INTERNAL USE ONLY', '', '', '', ''],
    ['', '', '', '', '', ''],
    ['', 'StudentID', 'FullName', 'Course_Interest', 'Contact_Status', 'Score']
], columns=['Spacer', 'StudentID', 'FullName', 'Course_Interest', 'Contact_Status', 'Score'])

# 4. Combine and export to Excel
final_df = pd.concat([header_rows, df_data], ignore_index=True)

# We export without pandas headers or index so our "messy" structure is preserved exactly
final_df.to_excel('Spreadsheets/messy_student_leads.xlsx', index=False, header=False)
print("File 'messy_student_leads.xlsx' generated successfully!")

File 'messy_student_leads.xlsx' generated successfully!


### The Practice: Writing Production-Grade Extraction Code

To handle Excel in Python, we rely on `pandas` powered by the `openpyxl` engine (for `.xlsx` files). Here is how we're gonna handle real-world extraction scenarios.

1. **Bypassing Human Formatting (Targeted Extraction)**

Business reports often have headers, footers, and floating summary tables. we must isolate the exact grid of data we need.

In [2]:
import pandas as pd

# The targeted extraction
raw_data = pd.read_excel(
    'Spreadsheets/messy_student_leads.xlsx', 
    skiprows=3,                # Skips rows 1, 2, and 3. Row 4 becomes the official header!
    usecols="B:F",             # Grabs only columns B through F (ignoring the blank Column A)
    dtype={'StudentID': str},  # Now this works because the header perfectly matches
    na_values=['TBD', 'N/A', 'Missing'] # Cleans up the messy statuses instantly
)

### **Schema 1: High-Performance Telemetry Log**

This simulates a raw data feed coming from automotive sensors during a track session. Sensor data is notorious for dropping packets, leading to missing values and misaligned timestamps.

**Target File Name:** `track_telemetry_raw.xlsx`

_The Goal:_ Practice handling time-series data, interpolating missing numeric values, and dealing with extreme outliers (sensor glitches).

*Column Names:* `Timestamp`(Datetime) , `Car_ID`(String), `Engine_RPM`(Integer), `Oil_Temp_C`(Float), `Sector_Time`(String) 

In [3]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Ensure reproducible randomness
np.random.seed(42)
random.seed(42)

num_rows = 150

# 1. Timestamp: Simulating dropped sensor connections
start_time = datetime(2026, 7, 14, 14, 0, 0)
timestamps = []
current_time = start_time
for _ in range(num_rows):
    timestamps.append(current_time)
    # Randomly skip 1 to 5 seconds to mimic dropped packets
    skip = random.choice([1, 1, 1, 2, 5])
    current_time += timedelta(seconds=skip)

# 2. Car_ID: Mixed casing to test string normalization
cars = ['GT-R', 'gt-r', 'V8-Supercharged', 'v8-supercharged', 'F1-W14', 'f1-w14']
car_ids = np.random.choice(cars, num_rows)

# 3. Engine_RPM: Injecting impossible hardware outliers
rpms = np.random.randint(4500, 8500, num_rows).tolist()
for idx in random.sample(range(num_rows), 5):
    rpms[idx] = -999 # Impossible negative RPM
for idx in random.sample(range(num_rows), 5):
    rpms[idx] = 45000 # Sensor glitch spike

# 4. Oil_Temp_C: Injecting missing values (NaNs)
oil_temps = np.random.uniform(90.0, 125.0, num_rows)
oil_temps = np.round(oil_temps, 1)
# Force 15 random rows to be missing data
for idx in random.sample(range(num_rows), 15):
    oil_temps[idx] = np.nan

# 5. Sector_Time: Mixed string formats requiring regex
sector_times = []
for _ in range(num_rows):
    base_seconds = random.uniform(80.0, 95.0)
    fmt_choice = random.choice(['colon', 'seconds', 'milli'])
    
    if fmt_choice == 'colon':
        mins = int(base_seconds // 60)
        secs = base_seconds % 60
        sector_times.append(f"{mins}:{secs:04.1f}") # Format: 1:24.5
    elif fmt_choice == 'seconds':
        sector_times.append(f"{base_seconds:.1f}s") # Format: 84.5s
    else:
        mins = int(base_seconds // 60)
        secs = int(base_seconds % 60)
        millis = int((base_seconds - int(base_seconds)) * 1000)
        sector_times.append(f"{mins:02d}:{secs:02d}:{millis:03d}") # Format: 01:24:500

# 6. Assemble the grid
df_data = pd.DataFrame({
    'Timestamp': timestamps,
    'Car_ID': car_ids,
    'Engine_RPM': rpms,
    'Oil_Temp_C': oil_temps,
    'Sector_Time': sector_times
})

# 7. Add human-readable, messy header rows to bypass
header_rows = pd.DataFrame([
    ['SILVERSTONE TRACK TELEMETRY EXPORT', '', '', '', ''],
    ['CONFIDENTIAL - RACE ENGINEERS ONLY', '', '', '', ''],
    ['Timestamp', 'Car_ID', 'Engine_RPM', 'Oil_Temp_C', 'Sector_Time']
], columns=['Timestamp', 'Car_ID', 'Engine_RPM', 'Oil_Temp_C', 'Sector_Time'])

# Combine and export without default pandas indexing
final_df = pd.concat([header_rows, df_data], ignore_index=True)
final_df.to_excel('Spreadsheets/track_telemetry_raw.xlsx', index=False, header=False)

print("File 'track_telemetry_raw.xlsx' generated successfully!")

File 'track_telemetry_raw.xlsx' generated successfully!


### **Schema 2: Web Scraped Product Database**

When I Scrap external web servers, The data actually I got would never be perfect, it'll give raw HTML strings, heavily poluted with UI elements like currency symbols and whitespace.

**Target File Name:** `scrapped_tech_invetory.csv`

_The Goal_: Practice advanced string manipulation, regular expressions(Regex), and reltional data mapping.

**Column Name** `SKU_Code`(String), `Product_Title`(String), `Raw_Price`(Float), `Availability`(Boolean), `Vendor_ID`(Integer)

_Messiness to Inject via Python_:

1. `SKU_Code`: Add hidden Whitespace characters (`\n`, `\t`) at the start or end of the strings.

2. `Product_Title`: Make some titles etremely long, containing unwanted HTML tags like `<br>` or `<strong>`.

3. `Raw_Price`: Generate as strings with mixed currencies (e.g., `$1,299.99`, `€450.00`, `Contact for Price`).

4. `Availability`: Use inconsistent text: `In Stock`, `True`, `1`, `Out of Stock`, `False`, `0`.

5. `Vendor_ID`: Make this a float (e.g., `104.0`) so I have to safely convert it to an integer without losing data.

In [4]:
import pandas as pd
import numpy as np
import random as rd

# Setting the seed to ensure reproducible randomness.
np.random.seed(31)
rd.seed(31)

# Total number of rows.
num_rows = 150

# let's generate the SKU_ID, with some additional strings to make data messy.
add_strings = ['\n', '\t', '', '', '', '','','','','',''] # Increasing weight of empty string to minimize data messyness.
sku_id = [f"{rd.choice(add_strings)}SKU-{rd.randint(1110, 9989)}{rd.choice(add_strings)}" for _ in range(num_rows)]

# let's generate the Product titles along with price.
# Taking the dictionary so my price and items would not be dismatched.
titles_with_price ={
    'NVDIA RTX-3050 GPU': 1135,
    'NVME-SSD 256GB': 345,
    'Intel i5 12450hx chip': 234,
    'NVDIA RTX-4050 GPU': 2352,
    'NVME-SSD 512GB<strong>': 3334, # Adding some extra strings to mess with the data
    '<br>DDR4-RAM 32GB</br>': 2334, # Also in here
    'DDR5-RAM 16GB': 3444,
    'Intel i7 13450hs chip': 3323,
}

name_list = list(titles_with_price.keys())
product_title = np.random.choice(name_list, num_rows)
print(name_list)
raw_price = []
currencies = ['$', '€', '£']
for name in product_title:
    raw_price.append(f"{rd.choice(currencies)}{titles_with_price[name]}")
print(raw_price)
# messing with my price.
for _ in range(40):
    rand_int = rd.randint(0, num_rows-1)
    raw_price[rand_int] = 'Contact For Price'

# Generating the Availability Status.
avail_status = ['In Stock', 'Out of Stock', 'True', 'False', '1', '0']
availability = np.random.choice(avail_status, num_rows)

# Generating the vendor ID
# Using Vectorization to generate and round the float numbers
vendor_ids = np.random.uniform(low=101, high=700.0, size=num_rows)
rounded_vendor_ids = np.round(vendor_ids, 1)

# Generating the grid Now.
data = pd.DataFrame({
    'SKU_ID': sku_id,
    'Product_Title': product_title,
    'Raw_Price': raw_price,
    'Availability': availability,
    'Vendor_ID': rounded_vendor_ids,
})

data.to_excel('Spreadsheets/scrapped_tech_invetory.xlsx', index=False, header=True)

print('File saved successfully!')


['NVDIA RTX-3050 GPU', 'NVME-SSD 256GB', 'Intel i5 12450hx chip', 'NVDIA RTX-4050 GPU', 'NVME-SSD 512GB<strong>', '<br>DDR4-RAM 32GB</br>', 'DDR5-RAM 16GB', 'Intel i7 13450hs chip']
['$234', '$3323', '€1135', '$3444', '£234', '€3323', '$234', '£3334', '$3334', '€2334', '$3444', '£3334', '$234', '€234', '€3444', '£3444', '£234', '$234', '€3444', '$345', '$345', '£3323', '€2352', '€3334', '£345', '€234', '€345', '£3334', '£234', '$3334', '€3323', '£2352', '$3444', '£2352', '£2352', '$234', '$1135', '£3323', '$3323', '$1135', '£3323', '£345', '£234', '£2352', '£3334', '$2352', '€3323', '£1135', '$3334', '€234', '$345', '€345', '€345', '$345', '€345', '€345', '£3444', '€3334', '$3334', '€3444', '£1135', '€3323', '£3444', '€345', '€1135', '€345', '£234', '£3323', '€2334', '$2352', '€345', '€3444', '£1135', '£2352', '£1135', '$2352', '$234', '€3334', '£345', '€234', '£234', '£345', '€1135', '€1135', '$2352', '£2352', '£1135', '£2334', '£2334', '£3444', '€2334', '€345', '£3334', '£2334', '£11

### **Creating an organized Workbook(Multisheets):**

The Standard `pd.to_excel('my_data.xlsx')` will just open an excel sheets, writes a single DataFrame and saved then closes the file, if we make it run again for different DataFrame , it just rewrite the previously saved sheets.

To keep Wite a Workbook having multiple sheets, I must have to keep the sheet open continously , which is done by `pd.ExcelWriter` . This holds the file open insert multiple dataframe before closing.

**SCHEMA: University Gradding pipline**

We will generate Three distinct table that relates to each other via Primary and Foreign key.

1. **Sheet1:** `Students`

| Student_ID | Full_Name | Enrollment_Year | Major |
|------------|-----------|-----------------|--------|
| STU-1101 | Sargio Ramos | 2024 | Commerce|
| STU-1102 | Virgil Vandigk| 2023 | Business|


`.` `.` `.`

2. **Sheet2:** `Course`

| Course_ID | Course_Name | Credit | Department |
|------------|-----------|-----------------|--------|
| CS-101 | DBMS | 4 | Engineering |
| BUS-201 | Enterpreneurship | 3 | Business|

`.` `.` `.`

3. **Sheet3:** `Grades`

| Record_ID | Student_ID | Course_ID | Semester |
|------------|-----------|-----------------|--------|
| REC-001 | STU-1001 | BUS-201 | Fall_2025 |
| REC-002 | STU-1001 | CS-101 | Spring_2025 |

`.` `.` `.`


In [2]:
import requests
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np

# load the environment variables.
load_dotenv()

api_key = os.getenv('X_API_KEY')

url = "https://api.api-ninjas.com/v1/babynames?gender=neutral"

# NOTE- The response.text are giving a list of only 10 names in the form of string
# e.g. "['alpha', 'romeo', 'beta', 'gamma']", so First I have to convert this into a python list
# And Since it's only 10 names I have to loop through the API Calls and extract almost 5 of them
# and merge them into a single list , so we'll have almost 50 names.

# NOTE- To Convert them into a standard python List we're gonna use a python built-in module
# called 'ast' (Abstract Syntax Tree) , along with a function called 'literal_eval()'
# literal_eval() evaluates the strings as Python literal expression. The primary use case for
# developers is safely converting stirngs into Python data types (list, dictonary, tuples,
# booleans etecetra.)
import ast
names = ['Emory', 'Ira', 'Brice', 'Tatum', 'Ravenel', 'Darren', 'Oakley', 'Arron', 'River', 'Lake', 'London', 'Carmine', 'Drew', 'Julian', 'Billy', 'Tatum', 'Hayden', 'Corey', 'Wynne', 'Phoenix', 'Sunny', 'Lucian', 'Ash', 'Finley', 'Dakota', 'Lee', 'Glen', 'Duncan', 'Baby', 'Harley', 'Kadin', 'Caden', 'Addison', 'Wesley', 'Shelby', 'Bay', 'Kendall', 'Moriah', 'Remy', 'Cameron', 'Lee', 'Phoenix', 'Shay', 'Charlie', 'Sage', 'Mackenzie', 'Ari', 'Francis', 'Cecil', 'Silver']

# NOTE-So Instead of I Call the API every single time , I just hardcoded the list of names.

# for _ in range(5):
#     try:
#         response = requests.get(url=url, headers={'X-Api-Key': api_key})
#         response.raise_for_status()
#         name_list = ast.literal_eval(response.text) # Convert the String output into List.
#         names += name_list        
#     except Exception as e:
#         print(f'Failed! {e}')

"""
NOW Let's build the Data Frame Since I got almost 50 different names.
"""
num_rows: int = 50

# Sheet 1
student_ids: list[str] = [f'STU-{num}' for num in range(1101, (1101 + num_rows))]
student_name: list[str] = names.copy() # A Shallow copy of names (list)
enrollment_years: list[int] = [2024, 2023, 2025, 2022] # Just created so I can choice random one of them.
enrollment_year: list[int] = np.random.choice(enrollment_years, num_rows)
major_subjects: list[str] = ['Comp. Sci', 'Finance', 'Commerce', 'Elect. Eng'] 
major: list[str] = np.random.choice(major_subjects, num_rows)

df_student: pd.DataFrame = pd.DataFrame({
    'Student_ID':student_ids,
    'Name': student_name,
    'Enrollment_Year': enrollment_year,
    'Major': major,
})

# Sheet 2 [Course_ID, Course_Name, Credit, Department]
course_description = {
    'CS-101': {'course_name': 'DBMS', 'credit': 4, 'department': 'Engineering'},
    'BUS-202': {'course_name': 'Entreprenuarship', 'credit': 3, 'department': 'Business'},
    'CS-102': {'course_name': 'Data Structures', 'credit': 4, 'department': 'Engineering'},
    'ELE-101': {'course_name': 'Circuits', 'credit': 3, 'department': 'Engineering'},
    'BUS-201': {'course_name': 'Finance', 'credit': 4, 'department': 'Business'},
    'ART-204': {'course_name': 'Painting', 'credit': 3, 'department': 'Arts'},
}

course_ids: list[str] = [course_id for course_id in course_description.keys()]
course_names: list[str] = [course_description[course_id]['course_name'] for course_id in course_ids]
credits: list[int] = [course_description[course_id]['credit'] for course_id in course_ids]
departments: list[str] = [course_description[course_id]['department'] for course_id in course_ids]

df_course: pd.DataFrame = pd.DataFrame({
    'Course_ID': course_ids,
    'Course_Name': course_names,
    'Credit': credits,
    'Department': departments,
})


# Sheet 3: [Record_ID, Student_ID, Course_ID, Semester]
record_id: list[str] = [f"REC-{id}" for id in range(101, (101 + num_rows))]
grades: float = np.round(np.random.uniform(59.0, 98.0, num_rows), 1)
semesters = ['Fall_2025', 'Spring_2025', 'Atumn_2025']

df_grades: pd.DataFrame = pd.DataFrame({
    'Record_ID': record_id,
    'Student_ID': student_ids,
    'Course_ID': np.random.choice(course_ids, num_rows),
    'Grades': grades,
    'Semester': np.random.choice(semesters, num_rows),
}) 

# Now we finished making multiple DataFrames, let's move on generating multisheets workbook in excel Using ExcelWriter
os.makedirs('Spreadsheets', exist_ok= True)
file_path: str = 'Spreadsheets/Multisheets_University_DBMS.xlsx'

# The 'With' statement ensures the file is safely closed after writing, even if the code crashes.
with pd.ExcelWriter(path=file_path, engine='openpyxl') as writer:

    # Write Each data frame to a specific sheetname
    df_student.to_excel(writer, sheet_name='Students', index=False)
    df_course.to_excel(writer, sheet_name='Course', index=False)
    df_grades.to_excel(writer, sheet_name='Grades', index=False)

print(f"Successfully generated multisheet Workbook at: {file_path}")

Successfully generated multisheet Workbook at: Spreadsheets/Multisheets_University_DBMS.xlsx
